# Geometry sanity check

Validate `src/umud/geometry.py` against real ground-truth masks before trusting it downstream of a trained model. Since no training image has *both* an aponeurosis mask and a fascicle mask, MT is checked directly against real apo masks; FL/PA are checked using a real fasc mask paired with an illustrative/synthetic apo line, since true paired ground truth doesn't exist.

Uses `binarize_mask()` (see `notebooks/01_eda.ipynb` §2) since apo and fasc masks use opposite raw-value polarity.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..") / "src"))

import numpy as np
import matplotlib.pyplot as plt

from umud.data import schema
from umud.data.dataset import load_image, load_mask, binarize_mask
from umud.geometry import measure_fascicle, measure_thickness

%matplotlib inline

## Muscle Thickness on real aponeurosis masks

In [ ]:
apo_files = schema.list_image_files(schema.APO_MASK_DIR)

for p in apo_files[:10]:
    mask = binarize_mask(load_mask(p)).astype(np.uint8)
    mt = measure_thickness(mask)
    print(f"{p.name}: MT = {mt:.1f} px")

If most values come back `nan`, `_largest_components` isn't finding two separate aponeurosis components -- check the mask visualization in `01_eda.ipynb` (masks might encode the two aponeuroses as one connected blob, or as more than two components).

## Fascicle Length / Pennation Angle on real fascicle masks (no paired apo mask -> angle measured against horizontal)

In [ ]:
fasc_files = schema.list_image_files(schema.FASC_MASK_DIR)

for p in fasc_files[:10]:
    mask = binarize_mask(load_mask(p)).astype(np.uint8)
    fl, pa = measure_fascicle(mask)
    print(f"{p.name}: FL = {fl:.1f} px, PA (vs. horizontal) = {pa:.1f} deg")

## Visual check: overlay the fitted line on top of the mask for a few examples

In [ ]:
from umud.geometry import _fit_line, _largest_components

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, p in zip(axes, fasc_files[:3]):
    mask = binarize_mask(load_mask(p)).astype(np.uint8)
    ax.imshow(mask, cmap="gray")
    components = _largest_components(mask, n=1)
    if components:
        centroid, direction = _fit_line(components[0])
        pts = components[0]
        proj = (pts - centroid) @ direction
        p0 = centroid + proj.min() * direction
        p1 = centroid + proj.max() * direction
        ax.plot([p0[1], p1[1]], [p0[0], p1[0]], "r-", linewidth=2)
    ax.set_title(p.name, fontsize=8)
plt.tight_layout()

## Findings checklist

- [ ] MT values look plausible (not mostly NaN, roughly consistent magnitude across samples)?
- [ ] Fitted fascicle line visually tracks the mask's principal axis?
- [ ] Any evidence of a pixel-to-mm scale factor anywhere in the data (revisit if found -- `pixel_to_mm` currently defaults to 1.0 everywhere)?